# Makes a table of the data from ghcnd_station_download_v1.0 and ENSO_indicies_download_v1.0

This script take daily observation data from station_data/ghcnd/all_station_data.csv  
and index data from indicies_data/first_pass/enso_historical_indicies.csv  
Then makes a built-out version of the table at the bottom of this page-> https://www.dnr.state.mn.us/climate/journal/el-nino-2026.html  

Note: averaging stations gives a 'station average' not necesarilly a 'Minnesota average'  
    more stations will decrease the difference between the two  

v0.0 designates this is a work in progress  
v1.0 should contribute to a reproducible workflow on any machine with the other 1.0 notebooks


In [1]:
import pandas as pd
from pathlib import Path
import nbformat

In [2]:
# uncomment and run the other two notebooks to make data if needed
# %run enso_indicies_download_v1.0.ipynb
# %run ghcnd_station_download_v1.0.ipynb

In [3]:
#assign data to vars for pandas
indicies_data=Path('../indicies_data/first_pass/enso_historical_indicies.csv')
station_data=Path("../station_data/ghcnd/all_station_data.csv")

#read data into the dataframe

#el nino index vals
indicies_data_df=pd.read_csv(
    indicies_data,
    skiprows=0,
    parse_dates=['Date']
)

#staion vals
stations_data_df=pd.read_csv(
    station_data,
    skiprows=0,
    parse_dates=['time']
)

#discard rows with no obs
stations_data_df=stations_data_df.dropna(
    subset=["TMAX", "TMIN", "PRCP", "MDPR", "DAPR",
               "SNOW", "MDSF", "DASF", "SNWD", "WESD", "WESF"],
    how='all'
)

In [4]:
#create a key they can be joined on
indicies_data_df['Month']=indicies_data_df['Date'].dt.to_period('M')
stations_data_df['Month']=stations_data_df['time'].dt.to_period('M')

In [5]:
#join the tables by date

station_obs_enso_indicies= pd.merge(
    left=stations_data_df,
    right=indicies_data_df,
    on='Month',
    how='left',
    validate='many_to_one'
)

In [6]:
#BEHOLD
store_table=Path('../outputs')
store_table.mkdir(parents=True, exist_ok=True)
station_obs_enso_indicies.to_csv(store_table/'station_obs_enso_indicies.csv', index=False)
#display(station_obs_enso_indicies)